In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json

from models import (PINN, PINNGELU, SIREN, HardBCPINN, exact_solution, source_term, laplacian, compute_loss)
from collocation import uniform_random, rad, rad_g, rar_g

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

with open('best_params.json') as f:
    P = json.load(f)

layers_tanh = [2] + [P['pinn_tanh']['width']] * P['pinn_tanh']['depth'] + [1]
layers_gelu = [2] + [P['pinn_gelu']['width']] * P['pinn_gelu']['depth'] + [1]
layers_siren = [2] + [P['siren']['width']] * P['siren']['depth'] + [1]
layers_hardbc = [2] + [P['hardbc']['width']] * P['hardbc']['depth'] + [1]


cuda


## Задача

Уравнение Пуассона на единичном квадрате с граничными условиями Дирихле:

$$\Delta u = f(x, y), \quad (x,y) \in [0,1]^2$$

Точное решение: $u = \sin(2\pi x)\sin(3\pi y)$, откуда $f = -13\pi^2 u$.

Ноутбук состоит из двух частей:
- **Часть 1** — сравнение архитектур (Tanh, GELU, SIREN, HardBC) при равномерной генерации точек
- **Часть 2** — сравнение методов коллокации (uniform, RAD, RAD-G, RAR-G) на SIREN


## Утилиты обучения

In [2]:
N_EPOCHS = 50000
PATIENCE = 3000
MIN_DELTA = 0.001
SEEDS = list(range(10))


def _eval_loss(model, eval_int, eval_bnd, w_b):
    loss, _, _ = compute_loss(model, eval_int, eval_bnd, 1.0, w_b)
    return loss.item()


def _eval_grid(model):
    """Rel. L2 на равномерной сетке 100x100."""
    x = torch.linspace(0, 1, 100)
    y = torch.linspace(0, 1, 100)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    grid = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)
    model.eval()
    with torch.no_grad():
        u_pred = model(grid).cpu().reshape(100, 100)
    u_exact = exact_solution(grid).cpu().reshape(100, 100)
    return (torch.norm(u_pred - u_exact) / torch.norm(u_exact)).item()


## Часть 1. Сравнение архитектур

Все модели обучаются с равномерной случайной генерацией внутренних точек.
Early stopping и выбор best_state происходит по eval loss на фиксированном наборе.
Проводим 10 независимых запусков с разным seed.


In [3]:
def train_arch(model_class, layers, lr, w_b, seed, **model_kwargs):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = model_class(layers, **model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    interior, boundary = uniform_random(1000, 400, device=device)
    eval_int, eval_bnd = uniform_random(1000, 400, device=device)

    best_loss = float('inf')
    best_state = None
    no_improve = 0

    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        loss, _, _ = compute_loss(model, interior, boundary, 1.0, w_b)
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            ev = _eval_loss(model, eval_int, eval_bnd, w_b)
            if ev < best_loss * (1 - MIN_DELTA):
                best_loss = ev
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                break

        if epoch % 1000 == 0 and epoch > 0:
            interior, _ = uniform_random(1000, 400, device=device)

    model.load_state_dict(best_state)
    return best_loss, _eval_grid(model)


In [4]:
arch_configs = [
    ('Tanh', PINN, layers_tanh, P['pinn_tanh']['lr'], P['pinn_tanh']['w_b'], {}),
    ('GELU', PINNGELU, layers_gelu, P['pinn_gelu']['lr'], P['pinn_gelu']['w_b'], {}),
    ('SIREN', SIREN, layers_siren, P['siren']['lr'], P['siren']['w_b'], {'omega_0': P['siren']['omega_0']}),
    ('HardBC', HardBCPINN, layers_hardbc, P['hardbc']['lr'], P['hardbc']['w_b'], {}),
]

arch_results = {name: {'loss': [], 'rel_l2': []} for name, *_ in arch_configs}

for name, cls, layers, lr, w_b, kwargs in arch_configs:
    print(name + ':', flush=True)
    for seed in SEEDS:
        loss, rl2 = train_arch(cls, layers, lr, w_b, seed, **kwargs)
        arch_results[name]['loss'].append(loss)
        arch_results[name]['rel_l2'].append(rl2)
        print('  seed {:2d}: loss={:.2e}  rel_l2={:.4f}'.format(seed, loss, rl2), flush=True)
    print(flush=True)


Tanh:


C:\Users\olegs\PycharmProjects\Calculating_Methods\.venv\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


KeyboardInterrupt: 

In [ ]:
print('{:<10} {:>12}  {:>12}  {:>12}'.format('Модель', 'Rel.L2 mean', 'Rel.L2 std', 'Loss mean'))
print('-' * 54)
for name in arch_results:
    rl2 = arch_results[name]['rel_l2']
    loss = arch_results[name]['loss']
    print('{:<10} {:>12.4f}  {:>12.4f}  {:>12.2e}'.format(name, np.mean(rl2), np.std(rl2), np.mean(loss)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(arch_results.keys())

axes[0].boxplot([arch_results[n]['rel_l2'] for n in names], tick_labels=names,
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[0].set_yscale('log')
axes[0].set_title('Rel. L2 Error')
axes[0].set_ylabel('Rel. L2 (log)')
axes[0].grid(True, alpha=0.3, which='both')

axes[1].boxplot([arch_results[n]['loss'] for n in names], tick_labels=names,
                patch_artist=True,
                boxprops=dict(facecolor='tomato', alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[1].set_yscale('log')
axes[1].set_title('Best Loss')
axes[1].set_ylabel('Loss (log)')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()


## Часть 2. Сравнение методов коллокации

Фиксируем архитектуру SIREN. Сравниваем четыре метода генерации внутренних точек:
- **uniform** — равномерный случайный (baseline)
- **RAD** — Residual-based Adaptive Distribution: сэмплируем пропорционально $\varepsilon^k$
- **RAD-G** — то же, но вес — норма градиента невязки $|\nabla_x r|$
- **RAR-G** — Residual-based Adaptive Refinement with Gradient: набор *растёт*, каждые 1000 эпох добавляем топ-50 точек по $|\nabla_x r|$

Early stopping для всех методов — по eval loss на фиксированном равномерном наборе.


In [ ]:
def poisson_residual(model, pts):
    """Невязка уравнения Пуассона: Delta u - f."""
    u = model(pts)
    return laplacian(u, pts) - source_term(pts)


In [ ]:
def train_colloc(method, seed, lr=None, w_b=None, omega_0=None):
    """Обучение SIREN с заданным методом коллокации."""
    if lr is None: lr = P['siren']['lr']
    if w_b is None: w_b = P['siren']['w_b']
    if omega_0 is None: omega_0 = P['siren']['omega_0']

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SIREN(layers_siren, omega_0=omega_0).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    interior, boundary = uniform_random(1000, 400, device=device)
    eval_int, eval_bnd = uniform_random(1000, 400, device=device)

    best_loss = float('inf')
    best_state = None
    no_improve = 0

    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        loss, _, _ = compute_loss(model, interior, boundary, 1.0, w_b)
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            ev = _eval_loss(model, eval_int, eval_bnd, w_b)
            if ev < best_loss * (1 - MIN_DELTA):
                best_loss = ev
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                break

        if epoch % 1000 == 0 and epoch > 0:
            if method == 'uniform':
                interior, _ = uniform_random(1000, 400, device=device)
            elif method == 'rad':
                interior = rad(model, 1000, poisson_residual, device=device)
            elif method == 'rad_g':
                interior = rad_g(model, 1000, poisson_residual, device=device)
            elif method == 'rar_g':
                interior = rar_g(model, interior, 50, poisson_residual, device=device)

    model.load_state_dict(best_state)
    return best_loss, _eval_grid(model)


In [ ]:
methods = ['uniform', 'rad', 'rad_g', 'rar_g']
colloc_results = {m: {'loss': [], 'rel_l2': []} for m in methods}

for method in methods:
    print(method + ':', flush=True)
    for seed in SEEDS:
        loss, rl2 = train_colloc(method, seed)
        colloc_results[method]['loss'].append(loss)
        colloc_results[method]['rel_l2'].append(rl2)
        print('  seed {:2d}: loss={:.2e}  rel_l2={:.4f}'.format(seed, loss, rl2), flush=True)
    print(flush=True)


In [ ]:
print('{:<10} {:>12}  {:>12}  {:>12}'.format('Метод', 'Rel.L2 mean', 'Rel.L2 std', 'Loss mean'))
print('-' * 54)
for m in methods:
    rl2 = colloc_results[m]['rel_l2']
    loss = colloc_results[m]['loss']
    print('{:<10} {:>12.4f}  {:>12.4f}  {:>12.2e}'.format(m, np.mean(rl2), np.std(rl2), np.mean(loss)))

labels = ['uniform', 'RAD', 'RAD-G', 'RAR-G']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot([colloc_results[m]['rel_l2'] for m in methods], tick_labels=labels,
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[0].set_yscale('log')
axes[0].set_title('Rel. L2 Error — методы коллокации, SIREN (10 seeds)')
axes[0].set_ylabel('Rel. L2 (log)')
axes[0].grid(True, alpha=0.3, which='both')

axes[1].boxplot([colloc_results[m]['loss'] for m in methods], tick_labels=labels,
                patch_artist=True,
                boxprops=dict(facecolor='tomato', alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[1].set_yscale('log')
axes[1].set_title('Best Loss — методы коллокации, SIREN (10 seeds)')
axes[1].set_ylabel('Loss (log)')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()
